In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import minimize, least_squares
import pccmnn as pc

# --- 0. 加载数据
csf_dict = pc.load_csf_data()

keys_to_delete = []

for key in csf_dict:
    sample = csf_dict[key]
    sample = sample[~np.isnan(sample).any(axis=1)]
    csf_dict[key] = sample
    
    if sample.shape[0] < 2:
        keys_to_delete.append(key)

for key in keys_to_delete:
    del csf_dict[key]

patient_data = [
    {'id': key, 'times': data[:,0], 'biomarkers': data[:,0:]}
    for key, data in csf_dict.items()
]

print('number of valid samples:', len(csf_dict))

# --- 1. 定义因果ODE模型 ---
# w 是一个包含21个群体参数的扁平化数组
def causal_ode_model(s, y, w):
    A, T, N, C = y
    
    # 提取参数 (w的顺序需要与论文Table 1对应)
    w_A0, w_A1, w_A2, \
    w_T0, w_T1, w_T2, w_T3, w_T4, w_T5, \
    w_N0, w_N1, w_N2, w_N3, w_N4, w_N5, \
    w_C0, w_C1, w_C2, w_C3, w_C4, w_C5 = w

    # ODE方程组 
    dAds = w_A0 + w_A1 * A + w_A2 * A**2
    dTds = w_T0 + w_T1 * T + w_T2 * T**2 + w_T3 * A + w_T4 * A * T + w_T5 * A**2
    dNds = w_N0 + w_N1 * N + w_N2 * N**2 + w_N3 * T + w_N4 * T * N + w_N5 * T**2
    dCds = w_C0 + w_C1 * C + w_C2 * C**2 + w_C3 * N + w_C4 * N * C + w_C5 * N**2
    
    return [dAds, dTds, dNds, dCds]

# --- 2. 辅助函数：根据参数求解ODE轨迹 ---
def get_model_trajectory(w, s_points):
    # 初始条件 
    y0 = [w[-1], 0, 0, 0] # 假设最后一个参数是y0, 即初始Abeta值
    s_span = [np.min(s_points), np.max(s_points)]
    
    # 使用solve_ivp求解
    sol = solve_ivp(
        fun=causal_ode_model,
        t_span=s_span,
        y0=y0,
        t_eval=s_points,
        args=(w[:-1],), # 传递除y0之外的w参数
        method='RK45'
    )
    return sol.y.T # 返回 (n_points, 4) 的数组

# --- 3. Population Model 校准算法 (Algorithm 1) ---

# 假设 patient_data 是一个列表，每个元素是一个字典，包含 'id', 'times', 'biomarkers'
# patient_data = [
#     {'id': 1, 'times': np.array([...]), 'biomarkers': np.array([[...],...])}, ...
# ]

def calibrate_population_model(patient_data, n_iterations=10):
    
    num_patients = len(patient_data)
    
    # --- 初始化 ---
    # 初始化群体参数 w (21个ODE参数 + 1个初始Abeta值 y0)
    w_population = np.random.rand(22) 
    
    # 初始化每个患者的DPS参数 (alpha, beta) 
    dps_params = {}
    for p in patient_data:
        alpha_init = np.random.uniform(0.1, 4.0)
        t_min = np.min(p['times'])
        t_max = np.max(p['times'])
        lower_bound_beta = -10 - alpha_init * t_min
        upper_bound_beta = 20 - alpha_init * t_max

        # 检查这个范围是否有效 (上界必须大于等于下界)
        if upper_bound_beta >= lower_bound_beta:
            # 从有效范围内选择一个值
            beta_init = np.random.uniform(lower_bound_beta, upper_bound_beta)
        else:
            beta_init = (lower_bound_beta + upper_bound_beta) / 2
        
        dps_params[p['id']] = np.array([alpha_init, beta_init])
        
    # --- 迭代优化 ---
    for it in range(n_iterations):
        print(f"--- Iteration {it + 1}/{n_iterations} ---")
        
        # --- 步骤 1: 固定DPS参数，优化群体参数 w --- [cite: 250]
        
        def objective_w(w):
            total_error = 0
            for p in patient_data:
                alpha, beta = dps_params[p['id']]
                s_points = alpha * p['times'] + beta
                
                # 获取模型预测
                # 需处理s_points非单调问题，先排序
                sort_idx = np.argsort(s_points)
                s_sorted, bio_sorted = s_points[sort_idx], p['biomarkers'][sort_idx]
                
                try:
                    y_pred = get_model_trajectory(w, s_sorted)
                    # 计算误差
                    error = np.sum((bio_sorted - y_pred)**2)
                    total_error += error
                except Exception as e:
                    # 惩罚无法求解的参数组合
                    return 1e12 
                regularization_term = 0.1 * np.sum(w[:-1]**2) # 對y0不做懲罰
            return total_error + regularization_term

        # 使用最小二乘法或minimize进行优化
        # 这里使用minimize作为示例
        w_bounds = [(-5.0, 5.0)] * 21 + [(1e-6, 1e-3)] # 為w和y0設置邊界
        res_w = minimize(objective_w, w_population, method='Nelder-Mead', bounds=w_bounds, options={'maxiter': 50})
        w_population = res_w.x
        print(f"Current loss: {res_w.fun}")

        # --- 步骤 2: 固定群体参数w，优化每个患者的DPS参数 (alpha, beta) --- [cite: 264]
        
        for p in patient_data:
            def objective_dps(dps_ab):
                alpha, beta = dps_ab
                s_points = alpha * p['times'] + beta
                
                # 需处理s_points非单调问题
                sort_idx = np.argsort(s_points)
                s_sorted, bio_sorted = s_points[sort_idx], p['biomarkers'][sort_idx]
                
                try:
                    y_pred = get_model_trajectory(w_population, s_sorted)
                    # 加权误差(这里简化为不加权) 
                    error = np.sum((bio_sorted - y_pred)**2)
                    return error
                except Exception as e:
                    return 1e12

            # 对每个患者进行独立的优化
            initial_guess = dps_params[p['id']]
            # 为alpha和beta设置边界，防止无解
            bounds = [(0.1, 4.0), (-100, 100)] 
            res_dps = minimize(objective_dps, initial_guess, method='L-BFGS-B', bounds=bounds)
            dps_params[p['id']] = res_dps.x

    return w_population, dps_params

# --- 运行 ---
w_final, dps_final = calibrate_population_model(patient_data)
print("Final Population Parameters (w):", w_final)
print("Final DPS Parameters:", dps_final)

number of valid samples: 251
--- Iteration 1/10 ---


ValueError: operands could not be broadcast together with shapes (22,) (21,) 